In [ ]:
import pandas as pd
import numpy as np
from churn_prediction.paths import INTERIM_CLI_DIR, PROCESSED_DIR

In [2]:
# 1. ĐỌC DỮ LIỆU ĐÃ MERGE
print("\n1. ĐỌC DỮ LIỆU")

df_merged = pd.read_parquet(INTERIM_CLI_DIR / 'merged_orders.parquet')
print(f"   Merged orders: {df_merged.shape}")


1. ĐỌC DỮ LIỆU
   Merged orders: (99441, 26)


In [ ]:
# 2. TẠO CUSTOMER-LEVEL
print("\n2. GROUPBY THEO CUSTOMER_ID")

print("\n2. GROUPBY THEO customer_unique_id (KHÁCH HÀNG THỰC)")

customer_df = df_merged.groupby('customer_unique_id').agg({
    # ===== IDENTIFIERS =====
    'customer_id': 'first',  # Lấy 1 customer_id đại diện
    'customer_city': 'first',
    'customer_state': 'first',
    'customer_zip_code_prefix': 'first',
    
    # ===== FREQUENCY FEATURES =====
    'order_id': 'count',  # num_orders
    'order_status': lambda x: (x == 'delivered').sum(),  # num_delivered
    
    # ===== MONETARY FEATURES =====
    'total_payment': 'sum',
    'total_price': 'sum',
    'total_freight': 'sum',
    'avg_price': 'mean',
    'avg_freight': 'mean',
    'max_installments': 'max',
    
    # ===== PRODUCT DIVERSITY =====
    'unique_products': 'sum',
    'unique_sellers': 'sum',
    'num_products': 'sum',
    
    # ===== REVIEW FEATURES =====
    'review_score': 'mean',
    'days_to_answer': 'mean',
    'num_comment_messages': 'sum',
    'num_comment_titles': 'sum',
    
    # ===== PAYMENT BEHAVIOR =====
    'main_payment_type': lambda x: x.mode()[0] if len(x) > 0 else 'unknown',
    
    # ===== TEMPORAL FEATURES =====
    'order_purchase_timestamp': ['min', 'max']
}).reset_index()

# Đặt lại tên cột
customer_df.columns = [
    'customer_unique_id', 'customer_id', 'city', 'state', 'zip_prefix',
    'num_orders', 'num_delivered',
    'total_payment', 'total_price', 'total_freight', 'avg_price', 'avg_freight', 'max_installments',
    'unique_products', 'unique_sellers', 'total_products',
    'avg_review_score', 'avg_days_to_answer', 'total_comments_msg', 'total_comments_title',
    'main_payment_type',
    'first_purchase_date', 'last_purchase_date'
]

print(f"   Customer_df shape: {customer_df.shape}")
print(f"   Số khách hàng thực: {len(customer_df):,}")


2. GROUPBY THEO CUSTOMER_ID

2. GROUPBY THEO customer_unique_id (KHÁCH HÀNG THỰC)
   Customer_df shape: (96096, 23)
   Số khách hàng thực: 96,096


In [ ]:
# 3. TẠO TEMPORAL FEATURES
print("\n3. TẠO TEMPORAL FEATURES")

# Chuyển đổi datetime
customer_df['first_purchase_date'] = pd.to_datetime(customer_df['first_purchase_date'])
customer_df['last_purchase_date'] = pd.to_datetime(customer_df['last_purchase_date'])

# Ngày tham chiếu cuối cùng
end_date = customer_df['last_purchase_date'].max()
print(f"   Ngày tham chiếu: {end_date.strftime('%Y-%m-%d')}")

# Tạo features thời gian
customer_df['days_since_last_purchase'] = (end_date - customer_df['last_purchase_date']).dt.days
customer_df['customer_lifetime_days'] = (customer_df['last_purchase_date'] - customer_df['first_purchase_date']).dt.days
customer_df['days_since_first_purchase'] = (end_date - customer_df['first_purchase_date']).dt.days

# Thời gian trung bình giữa các lần mua
customer_df['avg_days_between_orders'] = np.where(
    customer_df['num_orders'] > 1,
    customer_df['customer_lifetime_days'] / (customer_df['num_orders'] - 1),
    0
)

print(f"   Đã tạo 4 temporal features")


3. TẠO TEMPORAL FEATURES
   Ngày tham chiếu: 2018-10-17
   Đã tạo 4 temporal features


In [13]:
# 4. TẠO RATIO FEATURES
print("\n4. TẠO RATIO FEATURES")
# Tỉ lệ giao hàng thành công
customer_df['delivery_success_rate'] = customer_df['num_delivered'] / customer_df['num_orders']

# Tỉ lệ phí ship so với giá
customer_df['freight_to_price_ratio'] = customer_df['total_freight'] / customer_df['total_price']
customer_df['freight_to_price_ratio'] = customer_df['freight_to_price_ratio'].replace([np.inf, -np.inf], 0)

# Giá trị trung bình mỗi đơn
customer_df['avg_order_value'] = customer_df['total_payment'] / customer_df['num_orders']

# Tỉ lệ sản phẩm trên seller
customer_df['products_per_seller'] = customer_df['unique_products'] / customer_df['unique_sellers']
customer_df['products_per_seller'] = customer_df['products_per_seller'].replace([np.inf, -np.inf], 0)

# Tỉ lệ có comment
customer_df['comment_rate'] = customer_df['total_comments_msg'] / customer_df['num_orders']

print(f"   Đã tạo 5 ratio features")


4. TẠO RATIO FEATURES
   Đã tạo 5 ratio features


In [6]:
# 5. XỬ LÝ MISSING VÀ INF
print("\n5. XỬ LÝ MISSING VÀ INF")
# Kiểm tra missing trước khi xử lý
missing_before = customer_df.isnull().sum().sum()
print(f"   Missing trước xử lý: {missing_before}")

# Fill missing numeric
numeric_cols = customer_df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    customer_df[col] = customer_df[col].fillna(0)

# Fill missing categorical
categorical_cols = customer_df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    customer_df[col] = customer_df[col].fillna('unknown')

# Thay thế vô cùng bằng 0
customer_df = customer_df.replace([np.inf, -np.inf], 0)

# Kiểm tra missing sau xử lý
missing_after = customer_df.isnull().sum().sum()
print(f"   Missing sau xử lý: {missing_after}")



5. XỬ LÝ MISSING VÀ INF
   Missing trước xử lý: 3726
   Missing sau xử lý: 0


In [17]:
# 6. THỐNG KÊ CƠ BẢN
print("\n6. THỐNG KÊ CƠ BẢN")

# Số lượng khách hàng theo số lần mua
print(f"\n   Số lượng khách hàng:")
customers_1_order = (customer_df['num_orders'] == 1).sum()
customers_2_orders = (customer_df['num_orders'] == 2).sum()
customers_3_plus = (customer_df['num_orders'] >= 3).sum()

print(f"   - 1 order: {customers_1_order:,} ({customers_1_order/len(customer_df)*100:.1f}%)")
print(f"   - 2 orders: {customers_2_orders:,} ({customers_2_orders/len(customer_df)*100:.1f}%)")
print(f"   - 3+ orders: {customers_3_plus:,} ({customers_3_plus/len(customer_df)*100:.1f}%)")

# Giá trị trung bình
print(f"\n   Giá trị trung bình:")
print(f"   - Total payment: ${customer_df['total_payment'].mean():.2f}")
print(f"   - Avg order value: ${customer_df['avg_order_value'].mean():.2f}")
print(f"   - Avg review score: {customer_df['avg_review_score'].mean():.2f}")
print(f"   - Days since last purchase: {customer_df['days_since_last_purchase'].mean():.1f} days")

# Kiểm tra phân phối số lần mua
print(f"\n   Phân phối chi tiết số lần mua:")
order_counts = customer_df['num_orders'].value_counts().sort_index()
for n_orders in [1, 2, 3, 4, 5]:
    if n_orders in order_counts.index:
        count = order_counts[n_orders]
        print(f"   - {n_orders} orders: {count:,} customers ({count/len(customer_df)*100:.1f}%)")




6. THỐNG KÊ CƠ BẢN

   Số lượng khách hàng:
   - 1 order: 93,099 (96.9%)
   - 2 orders: 2,745 (2.9%)
   - 3+ orders: 252 (0.3%)

   Giá trị trung bình:
   - Total payment: $163.78
   - Avg order value: $158.76
   - Avg review score: 4.09
   - Days since last purchase: 287.7 days

   Phân phối chi tiết số lần mua:
   - 1 orders: 93,099 customers (96.9%)
   - 2 orders: 2,745 customers (2.9%)
   - 3 orders: 203 customers (0.2%)
   - 4 orders: 30 customers (0.0%)
   - 5 orders: 8 customers (0.0%)


In [18]:
# 7. PHÂN PHỐI THEO STATE
print("\n7. PHÂN PHỐI THEO STATE")

state_stats = customer_df.groupby('state').agg({
    'customer_id': 'count',
    'total_payment': 'sum',
    'avg_review_score': 'mean'
}).round(2).sort_values('customer_id', ascending=False)

state_stats.columns = ['num_customers', 'total_revenue', 'avg_review']
print(state_stats.head(10))


7. PHÂN PHỐI THEO STATE
       num_customers  total_revenue  avg_review
state                                          
SP             40296     5880368.99        4.18
RJ             12379     2115742.39        3.88
MG             11254     1843646.73        4.14
RS              5276      877781.99        4.14
PR              4881      794529.07        4.19
SC              3529      607384.77        4.08
BA              3277      607351.05        3.86
DF              2073      350692.00        4.08
ES              1964      323349.95        4.02
GO              1951      341697.27        4.04


In [21]:
# 8. LƯU DỮ LIỆU
print("\n8. LƯU DỮ LIỆU")
customer_df.to_parquet(PROCESSED_DIR / 'customer_level.parquet', index=False)
print(f"   Đã lưu customer_level.parquet")
print(f"   Shape: {customer_df.shape}")
print(f"   Memory: {customer_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Lưu danh sách features
feature_list = pd.DataFrame({
    'feature_name': customer_df.columns.tolist(),
    'data_type': customer_df.dtypes.astype(str).tolist(),
    'missing_count': customer_df.isnull().sum().tolist()
})
feature_list.to_csv(PROCESSED_DIR / 'customer_features_list.csv', index=False)
print(f" Đã lưu danh sách features")

print("CREATE CUSTOMER-LEVEL HOÀN TẤT!")
print(f"Output: {len(customer_df):,} customers x {len(customer_df.columns)} features")


8. LƯU DỮ LIỆU
   Đã lưu customer_level.parquet
   Shape: (96096, 32)
   Memory: 53.82 MB
 Đã lưu danh sách features
CREATE CUSTOMER-LEVEL HOÀN TẤT!
Output: 96,096 customers x 32 features
